In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from solver import FDMSolver

# Set visual style
plt.style.use('seaborn-v0_8-darkgrid')

# 1. Define the Double-Well Potential
def double_well(x, V0, a):
    # V0 controls barrier height, 'a' controls well separation
    return V0 * (x**2 - a**2)**2

# 2. Setup the Solver and Interactive Plotting Function
def plot_double_well(V0=2.0, a=1.5):
    solver = FDMSolver(x_min=-4.0, x_max=4.0, N=1000)
    
    # Create a wrapper function for the current slider values
    def current_V(x):
        return double_well(x, V0, a)
        
    energies, waves = solver.solve(current_V, num_states=4)
    
    plt.figure(figsize=(10, 6))
    
    # Plot the potential
    x_vals = solver.x
    plt.plot(x_vals, current_V(x_vals), 'k-', lw=2, label='V(x)')
    
    # Plot the first 4 wavefunctions (shifted by their energy levels)
    colors = ['blue', 'red', 'green', 'purple']
    for n in range(4):
        # Scale wavefunction for better visibility
        psi_scaled = waves[n] * 1.5 + energies[n]
        plt.plot(x_vals, psi_scaled, color=colors[n], label=f'E{n} = {energies[n]:.3f}')
        # Draw energy level lines
        plt.axhline(energies[n], color=colors[n], linestyle='--', alpha=0.3)

    plt.ylim(0, max(10, V0 * a**4 + 2))
    plt.title(f"Double-Well Potential (Barrier V0={V0:.1f}, Separation a={a:.1f})")
    plt.xlabel("Position (x)")
    plt.ylabel("Energy")
    plt.legend(loc='upper right')
    plt.show()

# 3. Connect the Sliders
widgets.interact(plot_double_well, 
                 V0=widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5, description='Barrier (V0):'),
                 a=widgets.FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1, description='Separation (a):'));

interactive(children=(FloatSlider(value=2.0, description='Barrier (V0):', max=10.0, step=0.5), FloatSlider(val…

In [2]:
# 4. Define the Kronig-Penney (Periodic Lattice) Potential
def kronig_penney(x, V0, well_width, barrier_width, num_wells=5):
    """Creates a finite periodic array of potential wells."""
    V = np.ones_like(x) * V0  # Start with everything at the barrier height
    period = well_width + barrier_width
    
    for i in range(num_wells):
        center = (i - num_wells // 2) * period
        in_well = np.abs(x - center) < (well_width / 2.0)
        V[in_well] = 0.0
    return V

# 5. Setup the Kronig-Penney Interactive Plot
def plot_lattice(V0=15.0, barrier_width=0.2):
    solver = FDMSolver(x_min=-5.0, x_max=5.0, N=1500)
    
    # Fixed well width for stability, users tweak the barriers
    well_width = 1.0 
    
    def current_V(x):
        return kronig_penney(x, V0, well_width, barrier_width, num_wells=5)
        
    # We solve for more states to see the "bands" form
    num_states = 10
    energies, waves = solver.solve(current_V, num_states=num_states)
    
    plt.figure(figsize=(12, 6))
    x_vals = solver.x
    plt.plot(x_vals, current_V(x_vals), 'k-', lw=1.5, label='V(x)')
    
    # Plot energy levels to visualize bands
    for n in range(num_states):
        color = 'blue' if n % 2 == 0 else 'red'
        plt.axhline(energies[n], color=color, linestyle='-', alpha=0.6)
        
        # Only plot a couple of wavefunctions to keep it clean
        if n in [0, num_states-1]:
            psi_scaled = waves[n] * 2.0 + energies[n]
            plt.plot(x_vals, psi_scaled, color='purple', alpha=0.5)

    plt.ylim(0, V0 + 5)
    plt.title(f"Kronig-Penney Lattice (Barrier V0={V0:.1f}, Width={barrier_width:.2f})")
    plt.xlabel("Position (x)")
    plt.ylabel("Energy")
    plt.show()

# 6. Connect the Sliders
widgets.interact(plot_lattice, 
                 V0=widgets.FloatSlider(value=15.0, min=5.0, max=50.0, step=1.0, description='Barrier (V0):'),
                 barrier_width=widgets.FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, description='Barrier Width:'));

interactive(children=(FloatSlider(value=15.0, description='Barrier (V0):', max=50.0, min=5.0, step=1.0), Float…